In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [ ]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "backend" / "data").exists() and (candidate / "frontend").exists():
            return candidate
    raise FileNotFoundError("Could not find the GradeScope project root from the current notebook path.")


PROJECT_ROOT = find_project_root(Path.cwd())
BACKEND_DIR = PROJECT_ROOT / "backend"
DATA_DIR = BACKEND_DIR / "data"
SUMMARIES_DIR = DATA_DIR / "summaries"
PROCESSED_DIR = DATA_DIR / "processed"
ASSETS_DIR = BACKEND_DIR / "assets"
CHARTS_DIR = ASSETS_DIR / "charts"

FINAL_DASHBOARD_PATH = PROCESSED_DIR / "gradescope_final_dashboard.csv"
GPA_SUMMARY_PATH = SUMMARIES_DIR / "scraped_gpa_summary.csv"

for folder in [ASSETS_DIR, CHARTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PATHS = {
    "project_root": PROJECT_ROOT,
    "final_dashboard": FINAL_DASHBOARD_PATH,
    "gpa_summary": GPA_SUMMARY_PATH,
    "charts": CHARTS_DIR,
}
PATHS

In [ ]:
def load_required_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No live {label} file found at {path}. Run portal sync/parser first.")
    try:
        return pd.read_csv(path)
    except Exception as error:
        raise RuntimeError(f"Could not load {label} from {path}: {error}") from error


raw_df = load_required_csv(FINAL_DASHBOARD_PATH, "dashboard")
gpa_df = load_required_csv(GPA_SUMMARY_PATH, "GPA summary")

print("Dashboard rows:", len(raw_df))
print("Loaded from:", FINAL_DASHBOARD_PATH)
print("GPA rows:", len(gpa_df))

In [ ]:
ATTENDANCE_SAFE_LINE = 80
PASSING_MARKS_LINE = 55

def short_subject(value: object) -> str:
    text = str(value)
    replacements = {
        "Software Construction and Development": "SCD",
        "Lab: Software Construction and Development": "SCD Lab",
        "Formal Methods in Software Engineering": "Formal Methods",
        "Artificial Intelligence": "AI",
        "Information Security": "InfoSec",
        "Professional Practices": "Pro Practices",
        "Web Engineering": "Web Eng",
        "Teachings of Holy Quran": "Quran",
    }
    for full, short in replacements.items():
        text = text.replace(full, short)
    return text


def clean_dashboard_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    numeric_columns = [
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "total_obtained_marks",
        "current_marks_percentage",
    ]

    for column in numeric_columns:
        if column not in df.columns:
            df[column] = 0
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)

    if "subject" not in df.columns:
        df["subject"] = "Unknown Subject"

    df["subject"] = df["subject"].fillna("Unknown Subject").astype(str)
    df["subject_short"] = df["subject"].apply(short_subject)

    if "total_obtained_marks" not in df.columns or df["total_obtained_marks"].eq(0).all():
        df["total_obtained_marks"] = (
            df["quiz_marks"] + df["assignment_marks"] + df["mid_marks"] + df["final_marks"]
        )

    if "current_marks_percentage" not in df.columns or df["current_marks_percentage"].eq(0).all():
        df["current_marks_percentage"] = df["total_obtained_marks"]

    if "grade" not in df.columns:
        df["grade"] = "Not Entered"
    df["grade"] = df["grade"].fillna("Not Entered").replace("", "Not Entered")

    if "reason" not in df.columns:
        df["reason"] = ""
    df["reason"] = df["reason"].fillna("")

    return df


df = clean_dashboard_data(raw_df)
df.head()

In [ ]:
def build_recommendation(row: pd.Series) -> str:
    notes = []

    if row["attendance_percentage"] < ATTENDANCE_SAFE_LINE:
        notes.append("Attendance is below the 80% safe line. Attend upcoming classes and avoid unnecessary absents.")

    if row["current_marks_percentage"] < PASSING_MARKS_LINE:
        notes.append("Marks are below the 55% passing line. Prioritize quizzes, assignments, mids, and final preparation.")

    if row["final_marks"] == 0:
        notes.append("Final marks are not entered yet, so the current percentage may change.")

    if row["total_absent"] >= 7:
        notes.append("Absence count is high. This subject should be monitored closely.")

    if not notes:
        notes.append("Subject looks stable. Maintain attendance and performance.")

    return " ".join(notes)


def apply_risk_rules(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["attendance_risk"] = np.where(
        df["attendance_percentage"] < ATTENDANCE_SAFE_LINE,
        "Below Safe Line",
        "Safe",
    )
    df["marks_risk"] = np.where(
        df["current_marks_percentage"] < PASSING_MARKS_LINE,
        "Below Passing Line",
        "Safe",
    )
    df["final_risk_status"] = np.where(
        (df["attendance_percentage"] < ATTENDANCE_SAFE_LINE)
        | (df["current_marks_percentage"] < PASSING_MARKS_LINE),
        "High Risk",
        "Safe",
    )
    df["recommendation"] = df.apply(build_recommendation, axis=1)

    return df


df = apply_risk_rules(df)
df[["subject", "attendance_percentage", "current_marks_percentage", "final_risk_status", "recommendation"]].head()

In [ ]:
def clean_gpa_data(gpa: pd.DataFrame) -> pd.DataFrame:
    if gpa.empty:
        raise ValueError("Live GPA summary is empty. Run portal sync/parser first.")

    gpa = gpa.copy()

    if "semester" not in gpa.columns:
        raise ValueError("Live GPA summary is missing the 'semester' column.")

    if "semester_order" not in gpa.columns:
        gpa["semester_order"] = range(1, len(gpa) + 1)

    if "semester_gpa" not in gpa.columns:
        raise ValueError("Live GPA summary is missing the 'semester_gpa' column.")

    gpa["semester_order"] = pd.to_numeric(gpa["semester_order"], errors="coerce").fillna(999999)
    gpa["semester_gpa"] = pd.to_numeric(gpa["semester_gpa"], errors="coerce").fillna(0)
    gpa = gpa.sort_values("semester_order")
    gpa["cumulative_gpa"] = gpa["semester_gpa"].expanding().mean().round(2)

    return gpa


gpa_df = clean_gpa_data(gpa_df)
gpa_df

In [ ]:
def calculate_kpis(df: pd.DataFrame, gpa_df: pd.DataFrame) -> dict:
    values = {
        "total_subjects": len(df),
        "high_risk_subjects": int((df["final_risk_status"] == "High Risk").sum()),
        "safe_subjects": int((df["final_risk_status"] == "Safe").sum()),
        "average_attendance": round(float(df["attendance_percentage"].mean()), 2) if len(df) else 0,
        "average_marks": round(float(df["current_marks_percentage"].mean()), 2) if len(df) else 0,
        "missing_finals": int((df["final_marks"] == 0).sum()) if "final_marks" in df.columns else 0,
        "total_absents": int(df["total_absent"].sum()) if "total_absent" in df.columns else 0,
        "latest_gpa": round(float(gpa_df["semester_gpa"].iloc[-1]), 2) if len(gpa_df) else 0,
        "cumulative_gpa": round(float(gpa_df["semester_gpa"].mean()), 2) if len(gpa_df) else 0,
    }
    return values


kpis = calculate_kpis(df, gpa_df)
kpi_df = pd.DataFrame([{"metric": key, "value": value} for key, value in kpis.items()])
kpi_df

In [ ]:
def save_plotly_figure(fig: go.Figure, name: str) -> None:
    html_path = CHARTS_DIR / f"{name}.html"
    fig.write_html(html_path)


def apply_white_theme(fig: go.Figure, height: int = 430) -> go.Figure:
    fig.update_layout(
        template="simple_white",
        height=height,
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        font=dict(color="#0f172a", size=13),
        margin=dict(l=40, r=40, t=50, b=50),
        legend_title="",
    )
    fig.update_xaxes(
        color="#0f172a",
        linecolor="#0f172a",
        gridcolor="#e5e7eb",
        zerolinecolor="#94a3b8",
        showline=True,
    )
    fig.update_yaxes(
        color="#0f172a",
        linecolor="#0f172a",
        gridcolor="#e5e7eb",
        zerolinecolor="#94a3b8",
        showline=True,
    )
    return fig

In [ ]:
fig = go.Figure()

labels = ["Subjects", "High Risk", "Safe", "Avg Attendance", "Avg Marks", "CGPA"]
values = [
    kpis["total_subjects"],
    kpis["high_risk_subjects"],
    kpis["safe_subjects"],
    kpis["average_attendance"],
    kpis["average_marks"],
    kpis["cumulative_gpa"],
]

fig.add_trace(
    go.Bar(
        x=labels,
        y=values,
        text=values,
        textposition="outside",
        marker=dict(color=["#2563eb", "#dc2626", "#16a34a", "#0891b2", "#7c3aed", "#d97706"]),
    )
)

fig.update_layout(title="GradeScope KPI Summary")
fig = apply_white_theme(fig, height=430)
save_plotly_figure(fig, "kpi_summary")
fig.show()

In [ ]:
fig = px.bar(
    df,
    x="subject_short",
    y="attendance_percentage",
    color="final_risk_status",
    text="attendance_percentage",
    color_discrete_map={"High Risk": "#dc2626", "Safe": "#16a34a"},
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.add_hline(
    y=ATTENDANCE_SAFE_LINE,
    line_dash="dash",
    line_color="#000000",
    line_width=2,
    annotation_text="80% safe line",
    annotation_position="top left",
)
fig.update_yaxes(range=[0, 110], title="Attendance %")
fig.update_xaxes(title="Subject")
fig.update_layout(title="Attendance Health")
fig = apply_white_theme(fig, height=440)
save_plotly_figure(fig, "attendance_health_pro")
fig.show()

In [ ]:
fig = px.bar(
    df,
    x="subject_short",
    y="current_marks_percentage",
    color="final_risk_status",
    text="current_marks_percentage",
    color_discrete_map={"High Risk": "#dc2626", "Safe": "#16a34a"},
)

fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
fig.add_hline(
    y=PASSING_MARKS_LINE,
    line_dash="dash",
    line_color="#000000",
    line_width=2,
    annotation_text="55% passing line",
    annotation_position="top left",
)
fig.update_yaxes(range=[0, 110], title="Marks %")
fig.update_xaxes(title="Subject")
fig.update_layout(title="Marks Performance")
fig = apply_white_theme(fig, height=440)
save_plotly_figure(fig, "marks_performance_pro")
fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=gpa_df["semester"],
        y=gpa_df["semester_gpa"],
        mode="lines+markers+text",
        name="SGPA",
        line=dict(width=4, shape="spline", color="#2563eb"),
        marker=dict(size=11, color="#2563eb", line=dict(color="#ffffff", width=2)),
        text=gpa_df["semester_gpa"].round(2),
        textposition="top center",
    )
)
fig.add_trace(
    go.Scatter(
        x=gpa_df["semester"],
        y=gpa_df["cumulative_gpa"],
        mode="lines+markers+text",
        name="CGPA",
        line=dict(width=4, shape="spline", color="#16a34a"),
        marker=dict(size=11, color="#16a34a", line=dict(color="#ffffff", width=2)),
        text=gpa_df["cumulative_gpa"].round(2),
        textposition="bottom center",
    )
)

fig.update_yaxes(range=[0, 4.1], title="GPA")
fig.update_xaxes(title="Semester")
fig.update_layout(title="SGPA and CGPA Trend")
fig = apply_white_theme(fig, height=450)
save_plotly_figure(fig, "gpa_trend")
fig.show()

In [ ]:
assessment_df = pd.DataFrame(
    {
        "component": ["Quiz", "Assignment", "Mid", "Final"],
        "marks": [
            df["quiz_marks"].sum(),
            df["assignment_marks"].sum(),
            df["mid_marks"].sum(),
            df["final_marks"].sum(),
        ],
    }
)

fig = px.pie(
    assessment_df,
    names="component",
    values="marks",
    hole=0.58,
    title="Assessment Contribution",
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig = apply_white_theme(fig, height=420)
save_plotly_figure(fig, "assessment_contribution")
fig.show()

In [ ]:
grade_df = df["grade"].fillna("Not Entered").replace("", "Not Entered").value_counts().reset_index()
grade_df.columns = ["grade", "count"]

fig = px.pie(
    grade_df,
    names="grade",
    values="count",
    hole=0.55,
    title="Grade Distribution",
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig = apply_white_theme(fig, height=420)
save_plotly_figure(fig, "grade_distribution")
fig.show()

In [ ]:
total_classes = df["total_present"] + df["total_absent"] + df["total_late"]
absence_pressure = (df["total_absent"] / total_classes.replace(0, np.nan) * 100).fillna(0)

health_df = pd.DataFrame(
    {
        "Subject": df["subject_short"],
        "Attendance %": df["attendance_percentage"],
        "Marks %": df["current_marks_percentage"],
        "Absence Pressure %": absence_pressure.round(2),
    }
)
metrics = ["Attendance %", "Marks %", "Absence Pressure %"]

fig = go.Figure(
    data=go.Heatmap(
        z=health_df[metrics].T.values,
        x=health_df["Subject"],
        y=metrics,
        colorscale="RdYlGn",
        text=health_df[metrics].T.round(1).values,
        texttemplate="%{text}",
        colorbar=dict(title="Value"),
    )
)
fig.update_layout(title="Subject Health Heatmap")
fig = apply_white_theme(fig, height=380)
save_plotly_figure(fig, "subject_health_heatmap")
fig.show()

In [ ]:
spread_df = pd.DataFrame(
    {
        "Attendance %": df["attendance_percentage"],
        "Marks %": df["current_marks_percentage"],
    }
).melt(var_name="metric", value_name="value")

fig = px.box(
    spread_df,
    x="metric",
    y="value",
    points="all",
    title="Performance Spread",
)
fig.update_yaxes(range=[0, 110], title="Percentage")
fig.update_xaxes(title="")
fig = apply_white_theme(fig, height=400)
save_plotly_figure(fig, "performance_spread")
fig.show()

In [ ]:
plot_df = df.copy()
duplicates = plot_df.groupby(["attendance_percentage", "current_marks_percentage"]).cumcount()

offset_x = [-1.5, -0.75, 0, 0.75, 1.5]
offset_y = [1.4, -1.4, 0, 2.0, -2.0]

plot_df["x"] = plot_df["attendance_percentage"] + duplicates.map(lambda index: offset_x[index % len(offset_x)])
plot_df["y"] = plot_df["current_marks_percentage"] + duplicates.map(lambda index: offset_y[index % len(offset_y)])
plot_df["size"] = (11 + plot_df["total_absent"].fillna(0) * 1.8).clip(11, 28)

fig = go.Figure()

for status, color in [("Safe", "#16a34a"), ("High Risk", "#dc2626")]:
    subset = plot_df[plot_df["final_risk_status"] == status]
    if subset.empty:
        continue

    fig.add_trace(
        go.Scatter(
            x=subset["x"],
            y=subset["y"],
            mode="markers+text",
            text=subset["subject_short"],
            textposition="middle right",
            name=status,
            marker=dict(
                size=subset["size"],
                color=color,
                opacity=0.88,
                line=dict(color="#ffffff", width=2),
            ),
            customdata=subset[["subject", "attendance_percentage", "current_marks_percentage", "total_absent"]],
            hovertemplate="<b>%{customdata[0]}</b><br>Attendance: %{customdata[1]:.1f}%<br>Marks: %{customdata[2]:.0f}%<br>Absents: %{customdata[3]}<extra></extra>",
        )
    )

fig.add_vline(x=ATTENDANCE_SAFE_LINE, line_dash="dash", line_color="#000000", line_width=2)
fig.add_hline(y=PASSING_MARKS_LINE, line_dash="dash", line_color="#000000", line_width=2)
fig.update_xaxes(range=[0, 110], title="Attendance %")
fig.update_yaxes(range=[-5, 110], title="Marks %")
fig.update_layout(title="Attendance vs Marks Risk Map")
fig = apply_white_theme(fig, height=430)
save_plotly_figure(fig, "risk_map_pro")
fig.show()

In [ ]:
clean_export_path = PROCESSED_DIR / "gradescope_notebook_clean_dashboard.csv"

export_columns = [
    "subject",
    "subject_short",
    "attendance_percentage",
    "total_present",
    "total_absent",
    "total_late",
    "quiz_marks",
    "assignment_marks",
    "mid_marks",
    "final_marks",
    "total_obtained_marks",
    "current_marks_percentage",
    "grade",
    "reason",
    "attendance_risk",
    "marks_risk",
    "final_risk_status",
    "recommendation",
]

available_columns = [column for column in export_columns if column in df.columns]
df[available_columns].to_csv(clean_export_path, index=False)
print(f"Clean notebook export saved to: {clean_export_path}")